# Natural language instruction generation

In [ ]:
import openai
import json
import os
from pathlib import Path

from openai import OpenAI

# Resolve paths from either the repository root or the dataset/ folder.
repo_root = Path.cwd()
if not (repo_root / "keys" / "keys.json").exists() and repo_root.name == "dataset":
    repo_root = repo_root.parent

keys_path = repo_root / "keys" / "keys.json"
with keys_path.open("r", encoding="utf-8") as f:
    keys = json.load(f)

client = OpenAI(api_key=keys["OPENAI_API_KEY"])
DATASET_DIR = repo_root / "dataset"

In [ ]:
# Load the cleaned behavior-tree dataset produced by bt_dataset_gen.ipynb.
dataset_path = DATASET_DIR / "merged_bt_dataset_cleaned.json"
with dataset_path.open("r", encoding="utf-8") as json_file:
    data = json.load(json_file)

In [ ]:
instruction = """
You are a helpful assistant that can help me with my tasks.
Below is an instruction that describes a task, paired with an input that provides further context. 
Write a response that appropriately completes the request.


"""

# Few-shot examples teach the model to summarize an XML behavior tree and list only action nodes.
example_user_prompt = """
You are given a behavior tree in XML format and your task is to summarize the task performed by this behavior tree as follows:
- Overall summary of the task.
- Use at most 200 words.
- Describe clearly in natural language.
- At the end of the summary, provide the list of action nodes present in the behavior tree, together with its parameters.
- Do not include in the list of action nodes the nodes that are not action nodes (e.g., Decorators, Sequence, Fallback, etc.).


<root main_tree_to_execute="MainTree">
  <BehaviorTree ID="MainTree">
    <PipelineSequence name="NavigateWithReplanning">
      <RateController hz="1.0">
        <ComputePathToPose goal="{goal}" path="{path}" planner_id="GridBased"/>
      </RateController>
      <FollowPath path="{path}"  controller_id="FollowPath"/>
    </PipelineSequence>
  </BehaviorTree>
</root>
"""

example_assistant_output= """
The behavior tree orchestrates the navigation of a robot by periodically replanning its global path at a frequency of 1 Hz. 
It utilizes a pipeline sequence, where it first computes a path to a specified goal using a "GridBased" planner and then follows this computed path using a designated controller. 
This approach ensures that the robot continuously updates its path to adapt to dynamic environments or changing conditions, enabling it to navigate effectively towards its goal while avoiding obstacles or other potential disruptions.

Actions: [ComputePathToPose (parameters: goal, path, planner_id), FollowPath (parameters: controller_id)]
"""

example_user_prompt_1 = """
You are given a behavior tree in XML format and your task is to summarize the task performed by this behavior tree as follows:
- Overall summary of the task.
- Use at most 200 words.
- Describe clearly in natural language.
- At the end of the summary, provide the list of action nodes present in the behavior tree.
- Do not include in the list of action nodes the nodes that are not action nodes (e.g., Decorators, Sequence, Fallback, etc.).


<root main_tree_to_execute = "MainTree" >
    <BehaviorTree ID="MainTree">
        <Sequence name="root_sequence">
            <Condition ID="CheckBattery"/>
            <Action    ID="OpenGripper"/>
            <Action    ID="ApproachObject"/>
            <Action    ID="CloseGripper"/>
        </Sequence>
    </BehaviorTree>
</root>
"""

example_assistant_output_1= """
The behavior tree outlines a robotic task sequence. 
First, it checks the battery level, then opens the robot's gripper, approaches an object, and finally closes the gripper. 
This sequence ensures that the robot performs these actions in order, with each action dependent on the success of the previous one. 
If any action fails, the subsequent actions will not be executed.

Actions: [OpenGripper, ApproachObject, CloseGripper]
"""

example_user_prompt_2 = """
You are given a behavior tree in XML format and your task is to summarize the task performed by this behavior tree as follows:
- Overall summary of the task.
- Use at most 200 words.
- Describe clearly in natural language.
- At the end of the summary, provide the list of action nodes present in the behavior tree.
- Do not include in the list of action nodes the nodes that are not action nodes (e.g., Decorators, Sequence, Fallback, etc.).


<root main_tree_to_execute = "MainSquare" >
    <BehaviorTree ID="MainSquare">
        <SequenceStar>
            <ArmTakeoff altitude="1.5"/>
            <SubTree ID="Square"/>
            <Land/>
        </SequenceStar>
    </BehaviorTree>
    <BehaviorTree ID="Square">
        <SequenceStar>
            <GoWaypoint name="Top Left" frame="1" x="4.0" y="0.0" z="-1.5" heading="0.0"/>
            <GoWaypoint name="Top Right" frame="1" x="4.0" y="4.0" z="-3.0" heading="0.0"/>
            <GoWaypoint name="Bottom Right" frame="1" x="0.0" y="4.0" z="-3.0" heading="0.0"/>
            <GoWaypoint name="Bottom Left" frame="1" x="0.0" y="0.0" z="-1.5" heading="0.0"/>
        </SequenceStar>
    </BehaviorTree>
 </root>
"""

example_assistant_output_2= """
The behavior tree orchestrates a drone to perform a square flight pattern at a specific altitude. 
The drone first takes off to an altitude of 1.5 units, then executes the "Square" sub-tree which guides it through four waypoints: Top Left, Top Right, Bottom Right, and Bottom Left, forming a square shape in a 3D space. 
The locations are top left (4.0, 0.0, -1.5), top right (4.0, 4.0, -3.0), bottom right (0.0, 4.0, -3.0), and bottom left (0.0, 0.0, -1.5).
Finally, the drone lands after completing the square flight pattern. 
This behavior is designed for automated control of the drone's flight path, allowing it to autonomously execute the specified sequence of actions.

Actions: [ArmTakeoff (parameters: altitude), Land, GoWaypoint (parameters: name, frame, x, y, z, heading)]
"""

user_prompt = """
You are given a behavior tree in XML format and your task is to summarize the task performed by this behavior tree as follows:
- Overall summary of the task.
- Use at most 200 words.
- Describe clearly in natural language.
- At the end of the summary, provide the list of action nodes present in the behavior tree.
- Do not include in the list of action nodes the nodes that are not action nodes (e.g., Decorators, Sequence, Fallback, etc.).
"""

In [ ]:
# Reload the cleaned BT source if this notebook is resumed from this section.
dataset_path = DATASET_DIR / "merged_bt_dataset_cleaned.json"
with dataset_path.open("r", encoding="utf-8") as json_file:
    data = json.load(json_file)

In [ ]:
from tqdm import tqdm

json_list = []
total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}

# Generate one natural-language task summary for each XML behavior tree.
for item in tqdm(data):
    bt = item["behavior_tree"]
    prompt = user_prompt + "\n\n" + bt

    try:
        completion = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": instruction},
                {"role": "user", "content": example_user_prompt},
                {"role": "assistant", "content": example_assistant_output},
                {"role": "user", "content": example_user_prompt_1},
                {"role": "assistant", "content": example_assistant_output_1},
                {"role": "user", "content": example_user_prompt_2},
                {"role": "assistant", "content": example_assistant_output_2},
                {"role": "user", "content": prompt}
            ],
            top_p=0.99,  # Keep summaries varied while preserving the requested format.
        )

        output = completion.choices[0].message.content
        json_list.append({
            "instruction": "System context",
            "input": output,
            "output": bt
        })

        # Track API usage so generation cost can be reproduced from the notebook run.
        usage = completion.usage
        total_usage["prompt_tokens"] += usage.prompt_tokens
        total_usage["completion_tokens"] += usage.completion_tokens
        total_usage["total_tokens"] += usage.total_tokens

    except openai.OpenAIError as e:
        print(f"OpenAIError while summarizing BT: {e}")
        continue

    except Exception as e:
        print(f"Unexpected error while summarizing BT: {e}")
        continue

print("Total Token Usage:")
print(json.dumps(total_usage, indent=4))

In [ ]:
# Save the instruction-format dataset before replacing the placeholder system prompt.
file_name = DATASET_DIR / "full_bt_dataset.json"
with file_name.open("w", encoding="utf-8") as json_file:
    json.dump(json_list, json_file, indent=4, ensure_ascii=False)

### Change instruction field

In [ ]:
# Load the generated instruction-format dataset for final prompt injection.
file_path = DATASET_DIR / "full_bt_dataset.json"
with file_path.open("r", encoding="utf-8") as file:
    data = json.load(file)

len(data)

In [ ]:
final_path = DATASET_DIR / "synthetic_bt_enhanced.json"

instruction = """
You are a helpful assistant that can assist with creating behavior trees.
Your task:
- Convert the provided summary of a behavior into an XML-formatted behavior tree.
- Ensure the behavior tree matches the description in the summary.
- The behavior tree must be compatible with the BehaviorTree.CPP library.
- Only use the actions and parameters provided in the action list below the summary.

Output Requirements:
- Output only the XML representation of the behavior tree. Do not include explanations, comments, or any additional text.
- Ensure all actions and parameters strictly match the provided list.
- If possible limit the use of SubTrees.

Please generate the behavior tree based on the summary and action list provided.
"""

# Replace the temporary placeholder with the final system prompt used for model training.
for item in data:
    item["instruction"] = instruction

with final_path.open("w", encoding="utf-8") as file:
    json.dump(data, file, indent=4, ensure_ascii=False)

### Check if dataset is clean

In [ ]:
import re

input_file = DATASET_DIR / "synthetic_bt_enhanced.json"
output_file = DATASET_DIR / "bt_dataset.json"


def clean_behavior_tree(xml_output):
    """Return exactly the XML behavior tree rooted at <root>...</root>."""
    match = re.search(r"<root[^>]*>.*?</root>", xml_output, re.DOTALL)
    if match:
        return match.group(0)
    return ""


def clean_dataset(input_file, output_file):
    # Clean only the assistant output field; instruction and input text are preserved.
    with input_file.open("r", encoding="utf-8") as file:
        data = json.load(file)

    for item in data:
        if "output" in item:
            item["output"] = clean_behavior_tree(item["output"])

    with output_file.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)


clean_dataset(input_file, output_file)
print(f"Dataset cleaned and saved to {output_file}.")

In [ ]:
import re

input_file = DATASET_DIR / "bt_dataset.json"


def contains_garbage(xml_output):
    """Return True when text appears before or after the <root>...</root> XML block."""
    match = re.search(r"<root[^>]*>.*?</root>", xml_output, re.DOTALL)
    if match:
        before_root = xml_output[:match.start()].strip()
        after_root = xml_output[match.end():].strip()
        return bool(before_root or after_root)
    return False


def find_garbage_indices(input_file):
    with input_file.open("r", encoding="utf-8") as file:
        data = json.load(file)

    # Print indices that still need manual inspection after automatic cleaning.
    for idx, item in enumerate(data):
        if "output" in item and contains_garbage(item["output"]):
            print(f"found at index {idx}")


find_garbage_indices(input_file)